# Supplementary tables

Assembles the supplementary table sheets from the canonical datasets and the analysis
outputs. Each is written to `chapters/06-supplementary-tables/sheets/` as a CSV;
`build_workbook.py` collects them into one workbook.

| Table | Built here | Source |
| --- | --- | --- |
| ST2 discordant lead variants | yes | Results 4 |
| ST4 gPS against gene sets | yes | Results 5 |
| ST5 ChEMBL target-indication pairs | yes | Results 6 |
| ST6 drug target enrichment | yes | Results 6 |
| ST7 PAV gene-disease pairs with 2-5 areas | yes | Results 6 |
| ST9 therapeutic area assignment | yes | the hierarchy itself |
| ST14 gene-disease associations with gPS | yes | data preparation |
| ST15 cluster membership | yes | data preparation |
| ST16 disease distribution across areas | yes | data preparation |
| ST1 studies, ST10 fine-mapping, ST11 colocalisation | no | need the release study and colocalisation datasets; `02_tables_from_release.ipynb` |
| ST3 GSEA, ST12 L2G performance, ST13 effector genes | no | blocked on missing inputs, see GAPS.md |
| ST8 subgroup analysis | no | needs the therapeutic-area and target-class breakdown of Results 6 |

In [ ]:
import pandas as pd
import pyarrow.dataset as ds

from manuscript_methods import clusters, paper

SHEETS = paper.ROOT / "chapters" / "06-supplementary-tables" / "sheets"
SHEETS.mkdir(parents=True, exist_ok=True)


def write(table: pd.DataFrame, name: str) -> None:
    """Write one sheet and report its shape."""
    path = SHEETS / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"{name}: {table.shape[0]} rows x {table.shape[1]} columns -> {path.name}")


names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

## ST9 — therapeutic area assignment

In [ ]:
st9 = pd.DataFrame(
    [{"EFO ID": root, "Therapeutic Area": label} for root, label in paper.THERAPEUTIC_AREAS.items()]
    + [{"EFO ID": "N/A", "Therapeutic Area": "other"}]
)
write(st9, "ST9_therapeutic_area_assignment")
st9

## ST2 — lead variants with discordant pleiotropic effects

In [ ]:
features = pd.read_parquet(paper.derived("variant_features"))
discordant = features[(features["uniqueDiseases"] >= 10) & (features["betaSignConcordance"] <= 0.8)].sort_values(
    "uniqueDiseases", ascending=False
)

st2 = pd.DataFrame(
    {
        "variantId": discordant["variantId"],
        "betaSignConcordance": discordant["betaSignConcordance"],
        "uniqueDiseases": discordant["uniqueDiseases"],
        "uniqueTherapeuticAreas": discordant["uniqueTherapeuticAreas"],
        "uniqueDiseaseNames": discordant["diseaseIds"].map(
            lambda ids: "; ".join(sorted({names.get(d, d) for d in (ids if ids is not None else [])}))
        ),
        "uniqueTherapeuticAreaNames": discordant["therapeuticAreas"].map(
            lambda tas: "; ".join(
                sorted({paper.THERAPEUTIC_AREAS.get(t, "other") for t in (tas if tas is not None else [])})
            )
        ),
        "prioritisedGenes": discordant["prioritisedGenes"].map(
            lambda genes: "; ".join(sorted(genes if genes is not None else []))
        ),
    }
)
write(st2, "ST2_discordant_variants")
st2.head()

## ST4 — gPS against membership in 21 gene sets

In [ ]:
st4 = pd.read_csv(paper.derived("gene_pleiotropy_by_category.csv"))
write(st4, "ST4_gPS_gene_categories")
st4.head()

## ST5 — all ChEMBL target-indication pairs with genetic support

In [ ]:
st5 = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))
write(st5, "ST5_chembl_ti_pairs")
print(
    "approved pairs:",
    int(st5["outcome"].sum()),
    "| approved with genetic support:",
    int(((st5["outcome"] == 1) & (st5["geneticSupport"] == 1)).sum()),
)
st5.head()

## ST6 — drug target enrichment results

In [ ]:
forest = pd.read_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"))
resources = pd.read_csv(paper.derived("drug_enrichment_other_resources.csv"))
st6 = pd.concat([forest, resources], ignore_index=True)
write(st6, "ST6_drug_target_enrichment")
st6[["datasource", "clinicalPhase", "odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3)

## ST7 — gene-disease associations supported by a PAV with 2 to 5 therapeutic areas

One row per credible set supporting such an association, as published.

In [ ]:
gene_table = pd.read_parquet(
    paper.derived("gene_table"), columns=["geneId", "approvedSymbol", "uniqueTherapeuticAreas"]
)
window = gene_table[gene_table["uniqueTherapeuticAreas"].between(2, 5)]

l2g = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"),
    columns=[
        "geneId",
        "studyLocusId",
        "studyId",
        "score",
        "eQTL_coloc",
        "pQTL_coloc",
        "VEP",
        "distanceTSS",
        "variantId",
        "maf",
        "absBeta",
        "diseaseIds",
        "year",
    ],
)
st7 = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(set(window["geneId"]))].merge(window, on="geneId", how="left")
write(st7, "ST7_pav_gene_disease_pairs")

pairs = st7[["geneId", "diseaseIds"]].explode("diseaseIds").dropna().drop_duplicates()
print("distinct gene-disease associations:", len(pairs), "(manuscript: 2734)")
st7.head(3)

## ST14 — every gene-disease association with gPS and area count

In [ ]:
gene_table = pd.read_parquet(paper.derived("gene_table"))
associations = (
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
    .explode("diseaseIds")
    .dropna()
    .drop_duplicates()
    .rename(columns={"diseaseIds": "diseaseId"})
)
st14 = associations.merge(
    gene_table[["geneId", "approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas"]], on="geneId", how="left"
)
st14["diseaseName"] = st14["diseaseId"].map(names)
st14["therapeuticArea"] = st14["diseaseId"].map(lambda d: paper.THERAPEUTIC_AREAS.get(areas.get(d, "other"), "other"))
st14 = st14.rename(columns={"uniqueDiseases": "gPS", "uniqueTherapeuticAreas": "numberOfTherapeuticAreas"})
write(st14, "ST14_gene_disease_with_gps")
st14.head()

## ST15 — diseases linked through each colocalisation cluster

In [ ]:
st15 = pd.read_parquet(paper.derived("cluster_membership"))
write(st15, "ST15_cluster_membership")
print("clusters:", st15["cluster_id"].nunique())
st15.head()

## ST16 — distribution of diseases across therapeutic areas

Two universes: every disease term carried by a qualifying study, and the gPS disease list,
which is every disease term carrying at least one credible set with an L2G-prioritised gene.

In [ ]:
qualifying_terms = set(
    ds.dataset(paper.derived("qualifying_gwas_studies"), format="parquet")
    .to_table(columns=["diseaseIds"])
    .to_pandas()["diseaseIds"]
    .explode()
    .dropna()
)
gps_terms = set(
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["diseaseIds"])["diseaseIds"]
    .explode()
    .dropna()
)
print("qualifying disease terms:", len(qualifying_terms), "| gPS disease terms:", len(gps_terms))


def counts(terms):
    """Disease terms per therapeutic area, measurements excluded."""
    labels = pd.Series([areas.get(t, "other") for t in terms])
    return labels[labels != paper.MEASUREMENT].value_counts()


qualifying_counts, gps_counts = counts(qualifying_terms), counts(gps_terms)
st16 = pd.DataFrame(
    {
        "Root ID": list(paper.THERAPEUTIC_AREAS) + ["other"],
        "Therapeutic Area": list(paper.THERAPEUTIC_AREAS.values()) + ["other (no area root)"],
    }
)
st16 = st16[st16["Root ID"] != paper.MEASUREMENT]
st16["Diseases (qualifying dataset)"] = st16["Root ID"].map(qualifying_counts).fillna(0).astype(int)
st16["Diseases (gPS list)"] = st16["Root ID"].map(gps_counts).fillna(0).astype(int)
for column in ["qualifying dataset", "gPS list"]:
    total = st16[f"Diseases ({column})"].sum()
    st16[f"% of {column}"] = (100 * st16[f"Diseases ({column})"] / total).round(1)
st16 = st16.sort_values("Diseases (gPS list)", ascending=False)
write(st16, "ST16_ta_distribution")
st16